# 🌧️ Coupled Satellite InSAR Ground Subsidence & EVT Extreme Rainfall Hazard Walkthrough
### Greater Accra Metropolitan Area (GAMA), Ghana — Hydro-Geomatic Hazard Intelligence
**Principal Investigator:** Peter Aryee  
**Package:** `accra_engine` (v1.0.0)

---

## Overview & Scientific Premise
This notebook demonstrates the end-to-end execution of the **Accra Hydro-InSAR Engine**:
1. **Atmospheric Extremes:** Modeling 1990–2024 daily rainfall via Extreme Value Theory (POT-GPD).
2. **Kinematic Satellite Radar:** Processing Sentinel-1 & NISAR multi-temporal InSAR displacement fields to resolve vertical subsidence rates ($v_{\text{vert}}$) up to $-35\text{ mm/yr}$.
3. **Topographic Geomorphology:** Evaluating DEM slope, Topographic Wetness Index (TWI), and drainage channels.
4. **Dynamic Hazard Coupling:** Computing the Dynamic Hydraulic Inundation Index (DHII) and training ML classifiers evaluated with AUC-PR and Matthews Correlation Coefficient (MCC).
5. **Decision Support:** Rendering interactive Folium Web GIS layers and 300 DPI diagnostic plots.

In [1]:
import sys
from pathlib import Path

# Ensure project src is in sys.path
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from accra_engine.data_ingest import AccraDataIngest
from accra_engine.evt_extremes import EVTExtremesEngine
from accra_engine.insar_deformation import InSARDeformationEngine
from accra_engine.coupled_hazard_ml import CoupledHazardEngine
from accra_engine.visualizer import AccraHazardVisualizer
from accra_engine.report_generator import HazardReportGenerator

print("✓ Accra Hydro-InSAR Engine v1.0.0 imported successfully.")

## 1. Data Ingestion: Historical Rainfall, InSAR Points & DEM Topography

In [2]:
data_dir = project_root / "data"
ingest = AccraDataIngest(cache_dir=data_dir)

# Ingest historical rainfall (1990 - 2024)
precip_rec = ingest.fetch_historical_rainfall(start_year=1990, end_year=2024)
print(f"Precipitation records: {precip_rec.n_records:,} daily observations")
print(f"Peak historical storm: {precip_rec.max_daily_precip:.1f} mm/day (3 June 2015)")

# Ingest InSAR Sentinel-1 displacement targets
insar_raw = ingest.load_or_generate_insar_data(n_points=2500, sensor="Sentinel-1_C-band", seed=42)
print(f"InSAR PS targets: {insar_raw.n_points:,} points (Wavelength: {insar_raw.wavelength_cm} cm)")

# Ingest DEM Hydrology Grid
dem_grid = ingest.load_or_generate_dem_grid(grid_resolution=0.005)
print(f"DEM Grid Shape: {dem_grid.shape} (Elevation: {dem_grid.elevation_m.min():.1f}m to {dem_grid.elevation_m.max():.1f}m)")

# Key infrastructure catalog
infra_df = ingest.get_key_infrastructure_assets()
infra_df.head()

## 2. Extreme Value Theory (EVT): Peaks-Over-Threshold & GPD MLE
We fit the Generalized Pareto Distribution (GPD):
$$F(y) = 1 - \left(1 + \frac{\xi y}{\sigma}\right)^{-1/\xi}$$
and compute design return levels for $T = [5, 10, 25, 50, 100]$ years with Delta-Method confidence intervals.

In [3]:
evt = EVTExtremesEngine(observations_per_year=365.25)
series_precip = precip_rec.df["precipitation_mm"]

# 1. Optimal threshold selection
u_opt = evt.suggest_optimal_threshold(series_precip)
print(f"Optimal POT threshold: u = {u_opt:.2f} mm")

# 2. Extract POT peaks
pot_res = evt.extract_pot(series_precip, threshold=u_opt, dates=precip_rec.df["date"], decluster_run_length=1)
print(f"POT exceedances: k = {pot_res.num_exceedances} extreme convective storm events")

# 3. Fit GPD via Maximum Likelihood Estimation
gpd_fit = evt.fit_gpd(pot_res)
print(f"GPD Scale sigma: {gpd_fit.scale:.3f} ± {gpd_fit.scale_se:.3f} mm")
print(f"GPD Shape xi:    {gpd_fit.shape:.4f} ± {gpd_fit.shape_se:.4f} (Frechet tail)")

# 4. Compute Return Levels
rl_res = evt.compute_return_levels(return_periods=[5.0, 10.0, 25.0, 50.0, 100.0], fit_result=gpd_fit, pot_result=pot_res)
print(rl_res.summary_df.to_string())

# 5. Goodness-of-Fit
gof = evt.test_goodness_of_fit(gpd_fit, pot_res, n_bootstraps=150)
print(f"KS test p-value: {gof.ks_pvalue:.4f} | AD test statistic: {gof.ad_statistic:.4f}")

## 3. InSAR Kinematics: LOS Decomposition & Subsidence Bowl Detection
Radar Line-of-Sight decomposition resolves vertical velocity:
$$v_{\text{vert}} = \frac{v_{\text{LOS}}}{\cos\theta_{\text{inc}}}$$
Filtering points at temporal coherence $\gamma \ge 0.35$ eliminates phase decorrelation noise.

In [4]:
insar_engine = InSARDeformationEngine(default_coherence_threshold=0.35)

# Coherence filtering
insar_filtered = insar_raw.filter_coherence(min_coherence=0.35)
insar_decomp = insar_engine.decompose_los_to_vertical(insar_filtered)

print(f"Peak ground subsidence: {insar_decomp['v_vert'].min():.1f} mm/yr")
print(f"Mean regional velocity:  {insar_decomp['v_vert'].mean():.1f} mm/yr")

# DBSCAN Subsidence Bowl Identification
bowls = insar_engine.identify_subsidence_bowls(
    insar_decomp,
    subsidence_threshold=-15.0,
    eps_km=0.85,
    min_samples=6,
)
bowl_df = pd.DataFrame([b.to_dict() for b in bowls])
print(bowl_df.to_string())

## 4. Coupled Multi-Hazard Machine Learning Engine
Coupling EVT 50-year rainfall ($185.2\text{ mm}$), InSAR ground subsidence, and DEM slope to compute the **Dynamic Hydraulic Inundation Index (DHII)**.

In [5]:
ml_engine = CoupledHazardEngine(model_type="hist_gradient_boosting", random_state=42)
r50 = float(rl_res.summary_df.loc[rl_res.summary_df["return_period_years"] == 50.0, "return_level_mm"].iloc[0])

coupled_df = ml_engine.build_coupled_dataset(
    insar_df=insar_decomp,
    rain_return_50yr=r50,
    dem_grid=dem_grid,
    seed=42,
)

# Train model & evaluate metrics
eval_res = ml_engine.train_and_evaluate(coupled_df, test_size=0.25)

print(f"Model: {eval_res.model_name.upper()}")
print(f"Accuracy:                              {eval_res.accuracy*100:.2f}%")
print(f"Matthews Correlation Coefficient (MCC): {eval_res.mcc:.4f}")
print(f"Precision-Recall AUC (AUC-PR):          {eval_res.auc_pr:.4f}")
print(f"ROC-AUC:                                {eval_res.auc_roc:.4f}")
print(f"Macro F1-Score:                         {eval_res.macro_f1:.4f}")

print("\nFeature Importances:")
for feat, imp in sorted(eval_res.feature_importances.items(), key=lambda x: x[1], reverse=True):
    print(f"  {feat:22s}: {imp*100:5.2f}%")

## 5. Decision Support: Web GIS Map & Publication Diagnostics
Outputs are rendered directly into the `output/` directory for ministerial decision-makers.

In [6]:
out_dir = project_root / "output"
vis = AccraHazardVisualizer(output_dir=out_dir)

# 1. Interactive Folium Map
map_path = vis.create_interactive_map(
    coupled_df=coupled_df,
    subsidence_bowls=bowls,
    infrastructure_df=infra_df,
)
print(f"Generated Leaflet Web GIS Map: {map_path}")

# 2. 300 DPI Publication 4-Panel Plot
plot_path = vis.generate_publication_diagnostics(
    precip_record=precip_rec,
    pot_result=pot_res,
    gpd_fit=gpd_fit,
    return_levels=rl_res,
    coupled_df=coupled_df,
    subsidence_bowls=bowls,
    infrastructure_df=infra_df,
)
print(f"Generated 300 DPI Diagnostics:  {plot_path}")

# 3. Technical Engineering Reports
rep_gen = HazardReportGenerator(output_dir=out_dir)
md_path, csv_path = rep_gen.generate_all_reports(
    pot_result=pot_res,
    gpd_fit=gpd_fit,
    return_levels=rl_res,
    gof_result=gof,
    subsidence_bowls=bowls,
    ml_eval=eval_res,
    coupled_df=coupled_df,
    infrastructure_df=infra_df,
)
print(f"Generated Vulnerability CSV:    {csv_path}")
print(f"Generated Engineering Report:   {md_path}")

## 6. Critical Infrastructure Exposure Summary
Audited facilities prioritized for structural reinforcement and stormwater retention chambers:

In [7]:
v_df = pd.read_csv(csv_path)
print(v_df.sort_values(by=["hazard_class_code", "criticality_score_10"], ascending=False).to_string())